### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B3819C5400>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B3819C5E80>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions and nested structures.

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description = "The title of the movie")
    year:int = Field(description = "This year the movie was released")
    director:str = Field(description = "The director of the movie")
    rating:float = Field(description = "The movie ratings out of 10")

In [6]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000019A1B984D70>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000019A1B9857F0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title':

In [7]:
model.invoke("Provide details about the movie Inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. Directed by Christopher Nolan, right? It\'s a sci-fi action film that came out in 2010. The main character is Dom Cobb, played by Leonardo DiCaprio. He\'s a thief who steals secrets by entering the subconscious of his targets. But instead of stealing, he\'s offered a chance to erase his criminal past by performing the opposite: planting an idea into someone\'s mind, which is called "inception."\n\nThe cast includes a lot of big names. Let me see... There\'s Joseph Gordon-Levitt, who plays Arthur, Cobb\'s right-hand man. Ellen Page is Ariadne, the architect who designs the dream. Tom Hardy is Balthazar "Bale" Selznick, the rival. Tom\'s character is a bit of a thorn in Cobb\'s side. Then there\'s Dileep Rao as the projectionist and Marion Cotillard as Mal, Cobb\'s wife, whose presence haunts him in the dreams due to her suicide in the real world.\n\nT

In [8]:
response = model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [9]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details"""
    title: str = Field(..., description = "The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw = True)

response = model_with_structure.invoke("Provide details about the movie Inception")
response


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me see what I need to do. The available tool is the Movie function, which requires title, year, director, and rating. I need to provide those parameters. \n\nFirst, I know that Inception is directed by Christopher Nolan. The title is "Inception". The year it was released was 2010. The rating is a bit tricky, but I recall it has a high rating on IMDb, maybe around 8.8. Let me confirm that. Yes, IMDb gives it 8.8/10. So putting that all together, I can structure the function call with those details. I need to make sure all required parameters are included and in the correct format. The year should be an integer, the rating a number, and the director\'s name as a string. Everything looks good. Let me format the JSON accordingly.\n', 'tool_calls': [{'id': 'zzgvwcvy1', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Incep

### Nested Structure

In [6]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description = "Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Arnie'), Actor(name='Tom Hardy', role="Cobb's Forger")], genres=['Science Fiction', 'Action', 'Thriller'], budget=160.0)

### TypedDict

TypeDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [4]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_typedict = model.with_structured_output(MovieDict)
response = model_with_typedict.invoke("Please provide the details of the movie Avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [ ]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float 

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man'},
  {'name': 'Chris Evans', 'role': 'Captain America'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Jeremy Renner', 'role': 'Hawkeye'}],
 'genres': ['Action', 'Science Fiction', 'Adventure'],
 'title': 'Avengers',
 'year': 2012}

In [9]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

Structured output has no attribute of profile but basic model has

In [ ]:
model_with_typedict.profile

### DataClasses

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator.

In [12]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

class ContactInfo(BaseModel):
    """Base information for a person"""
    name: str = Field(description = "The name of the person")
    email: str = Field(description = "The email address of the person")
    phone: str = Field(description = "The phone number of the person")

llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash-lite",
    google_api_key = os.getenv("GOOGLE_API_KEY")
)

agent = create_agent(
    model = llm,
    response_format = ContactInfo
)

result = agent.invoke({
    "messages":[{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [13]:
result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='f3918d07-5489-4aa6-b154-fb5cd67ebfad'),
  AIMessage(content='{\n  "name": "John Doe",\n  "email": "john@example.com",\n  "phone": "(555) 123-4567"\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f2061-f076-7dd3-9af8-520d4b727469-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 44, 'total_tokens': 73, 'input_token_details': {'cache_read': 0}})],
 'structured_response': ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')}

In [14]:
#using typedDict

from typing_extensions import TypedDict
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

class ContactInfo(TypedDict):
    """Base information for a person"""
    name: str 
    email: str 
    phone: str 

llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash-lite",
    google_api_key = os.getenv("GOOGLE_API_KEY")
)

agent = create_agent(
    model = llm,
    response_format = ContactInfo
)

result = agent.invoke({
    "messages":[{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


In [17]:
# Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information of a person"""
    name: str
    email: str
    phone: str

agent = create_agent(
    model = llm,
    response_format = ContactInfo
)

result = agent.invoke({
    "messages":[{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')
